# 10 - Reservation Objective Functions

This notebook defines a few scoring rules for strict Class 1 reservation and compares them to pooled FCFS on a small matched example.

Each rule turns one simulation run into one number. A higher number means the policy looks better under that rule. The rules here answer different practical questions:

- Are more slots used by the class we care about?
- Are more patients served after giving Class 1 extra weight?
- Does strict reservation still look good after charging a cost for protected slots?
- Does it still look good after also charging a cost for longer waits?
- Does it pass basic requirements, like using at least half the slots and not letting the two classes get too far apart?

Here `Q = reserved_slots_per_day` is the number of protected Class 1 slots per day.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")

## Objective Parameters

Class 2 always has weight 1.0. Class 1 gets the weights below. A larger Class 1 weight means we care more about serving Class 1 or using slots for Class 1. The slot-cost values say how much we penalize reserving a larger share of the day.


In [ ]:
CLASS_1_WEIGHTS = [1.0, 1.5, 2.0, 3.0]
CLASS_2_WEIGHT = 1.0
SLOT_COSTS = [0.0, 0.01, 0.02, 0.05]
WAIT_PENALTIES = [0.0, 0.02, 0.05]

UTILIZATION_FLOOR = 0.50
SERVED_RATE_FLOORS = [0.50, 0.60, 0.70]
CLASS_UTILIZATION_GAP_LIMITS = [0.10, 0.20, 0.30]

pd.DataFrame(
    {
        "class_1_weights": pd.Series(CLASS_1_WEIGHTS),
        "slot_costs": pd.Series(SLOT_COSTS),
        "wait_penalties": pd.Series(WAIT_PENALTIES),
        "served_rate_floors": pd.Series(SERVED_RATE_FLOORS),
        "class_utilization_gap_limits": pd.Series(CLASS_UTILIZATION_GAP_LIMITS),
    }
)

## Small Example Setup

This notebook uses a small example so the formulas can be checked quickly. The larger sweeps are in notebook 11.


In [ ]:
BASE_SCENARIO = {
    "slots_per_day": 16,
    "reserved_slots_per_day": 4,
    "reserved_class_id": 1,
    "horizon_days": 10,
    "burn_in_days": 10,
    "measure_days": 90,
    "cooldown_days": 10,
    "seeds": list(range(5101, 5111)),
    "classes": {
        1: {
            "lambda_per_day": 10.0,
            "cancel_prob": 0.08,
            "value": 1.0,
            "balk_prob": {"threshold": 6, "low": 0.00, "high": 0.45},
            "no_show_prob": {"threshold": 5, "low": 0.03, "high": 0.25},
        },
        2: {
            "lambda_per_day": 10.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 6, "low": 0.00, "high": 0.45},
            "no_show_prob": {"threshold": 5, "low": 0.03, "high": 0.25},
        },
    },
}


def build_config(scenario: dict, *, q: int, seed: int | None) -> SimulationConfig:
    classes = {}
    for class_id, params in scenario["classes"].items():
        classes[class_id] = PatientClassParams(
            class_id=class_id,
            lambda_per_day=float(params["lambda_per_day"]),
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=float(params["cancel_prob"]),
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=float(params.get("value", 1.0)),
        )

    return SimulationConfig(
        slots_per_day=int(scenario["slots_per_day"]),
        horizon_days=int(scenario["horizon_days"]),
        burn_in_days=int(scenario["burn_in_days"]),
        measure_days=int(scenario["measure_days"]),
        cooldown_days=int(scenario["cooldown_days"]),
        classes=classes,
        seed=seed,
        reserved_class_id=scenario["reserved_class_id"] if q > 0 else None,
        reserved_slots_per_day=int(q),
    )


pd.Series({key: value for key, value in BASE_SCENARIO.items() if key not in {"classes", "seeds"}}, name="value").to_frame()

## Run Summaries

Each simulation run is turned into one row with the totals needed for the scoring rules: arrivals, served patients, waiting time, slot use, and losses.


In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else 0.0


def result_summary_row(result, fixed_values: dict) -> dict:
    aggregate = aggregate_result_row(result, fixed_values)
    class_df = pd.DataFrame(class_result_rows(result, fixed_values)).set_index("class_id")
    c1 = class_df.loc[1]
    c2 = class_df.loc[2]
    total_arrivals = aggregate["total_arrivals"]
    total_offered = aggregate["total_offered"]

    return {
        **fixed_values,
        "slots_per_day": result.total_slots / result.slot_metrics.measured_days,
        "horizon_days": result.final_full_state and len(result.final_full_state) or np.nan,
        "total_arrivals": total_arrivals,
        "total_served": aggregate["total_served"],
        "total_offered": total_offered,
        "served_rate": safe_divide(aggregate["total_served"], total_arrivals),
        "average_utilization": aggregate["average_utilization"],
        "mean_offered_booking_delay": aggregate["mean_offered_booking_delay"],
        "total_balked": aggregate["total_balked"],
        "total_no_offer": aggregate["total_no_offer"],
        "total_canceled": aggregate["total_canceled"],
        "total_no_show": aggregate["total_no_show"],
        "total_unresolved_booked": aggregate["total_unresolved_booked"],
        "class_1_arrivals": c1["arrivals"],
        "class_2_arrivals": c2["arrivals"],
        "class_1_served": c1["served"],
        "class_2_served": c2["served"],
        "class_1_offered": c1["offered"],
        "class_2_offered": c2["offered"],
        "class_1_served_rate": c1["percent_serviced"],
        "class_2_served_rate": c2["percent_serviced"],
        "min_class_served_rate": min(c1["percent_serviced"], c2["percent_serviced"]),
        "class_1_slot_utilization": c1["slot_utilization"],
        "class_2_slot_utilization": c2["slot_utilization"],
        "class_utilization_gap": abs(c1["slot_utilization"] - c2["slot_utilization"]),
        "class_1_total_offered_delay": c1["total_offered_booking_delay"],
        "class_2_total_offered_delay": c2["total_offered_booking_delay"],
    }


def validate_accounting(row: pd.Series) -> None:
    partition = (
        row["total_served"]
        + row["total_balked"]
        + row["total_no_offer"]
        + row["total_canceled"]
        + row["total_no_show"]
        + row["total_unresolved_booked"]
    )
    if abs(partition - row["total_arrivals"]) > 1e-9:
        raise AssertionError("Outcomes do not partition arrivals.")

## Objective Functions

The formulas below are the comparison rules. Each one gives one number for FCFS and one number for strict reservation.

Notation used below: $S$ is slots per day, $T$ is measured days, $Q$ is protected Class 1 slots per day, $A_i$ is arrivals, $Y_i$ is served patients, and $D_i$ is total offered waiting days for class $i$. Class slot use is

$$
u_i = \frac{Y_i}{S T}.
$$

The weighted slot-use rule asks: how much of the calendar was used by each class, after giving Class 1 more or less importance?

$$
U_{slot}(w_1,w_2) = w_1 u_1 + w_2 u_2.
$$

The weighted served-rate rule asks: out of weighted demand, what share got served?

$$
U_{served}(w_1,w_2) = \frac{w_1 Y_1 + w_2 Y_2}{w_1 A_1 + w_2 A_2}.
$$

The slot-cost rule asks: does the policy still look good after charging a cost for protecting slots?

$$
U_{net}(w_1,w_2,c) = U_{served}(w_1,w_2) - c\frac{Q}{S}.
$$

The wait-adjusted rule asks: does the policy still look good after also charging a cost for longer offered waits? First compute the weighted average offered wait:

$$
\bar d_w = \frac{w_1 D_1 + w_2 D_2}{w_1 O_1 + w_2 O_2},
$$

where $O_i$ is the number of offered patients. Then subtract a wait penalty:

$$
U_{wait}(w_1,w_2,c,\gamma) = U_{slot}(w_1,w_2) - c\frac{Q}{S} - \gamma\frac{\bar d_w}{H}.
$$

The rule-based version asks a different question: among policies that pass basic requirements, which one has the best slot use or shortest wait? The requirements are:

$$
\rho \ge 0.50, \qquad \min_i \frac{Y_i}{A_i} \ge x, \qquad |u_1-u_2| \le g.
$$


In [ ]:
def weighted_mean_offered_delay(row: pd.Series, class_1_weight: float, class_2_weight: float = CLASS_2_WEIGHT) -> float:
    weighted_delay = (
        class_1_weight * row["class_1_total_offered_delay"]
        + class_2_weight * row["class_2_total_offered_delay"]
    )
    weighted_offered = (
        class_1_weight * row["class_1_offered"]
        + class_2_weight * row["class_2_offered"]
    )
    return safe_divide(weighted_delay, weighted_offered)


def score_rows_for_run(row: pd.Series) -> list[dict]:
    rows = []
    q_share = safe_divide(row["Q"], row["slots_per_day"])

    rows.append({**row.to_dict(), "score_name": "served_rate", "class_1_weight": 1.0, "slot_cost": 0.0, "wait_penalty": 0.0, "score_value": row["served_rate"]})

    for weight in CLASS_1_WEIGHTS:
        weighted_served_rate = safe_divide(
            weight * row["class_1_served"] + CLASS_2_WEIGHT * row["class_2_served"],
            weight * row["class_1_arrivals"] + CLASS_2_WEIGHT * row["class_2_arrivals"],
        )
        weighted_slot_score = (
            weight * row["class_1_slot_utilization"]
            + CLASS_2_WEIGHT * row["class_2_slot_utilization"]
        )
        weighted_delay = weighted_mean_offered_delay(row, weight)

        rows.append({**row.to_dict(), "score_name": "weighted_served_rate", "class_1_weight": weight, "slot_cost": 0.0, "wait_penalty": 0.0, "score_value": weighted_served_rate})
        rows.append({**row.to_dict(), "score_name": "weighted_slot_score", "class_1_weight": weight, "slot_cost": 0.0, "wait_penalty": 0.0, "score_value": weighted_slot_score})

        for slot_cost in SLOT_COSTS:
            rows.append({**row.to_dict(), "score_name": "net_priority_score", "class_1_weight": weight, "slot_cost": slot_cost, "wait_penalty": 0.0, "score_value": weighted_served_rate - slot_cost * q_share})

            for wait_penalty in WAIT_PENALTIES:
                delay_cost = wait_penalty * safe_divide(weighted_delay, row["horizon_days"])
                rows.append({**row.to_dict(), "score_name": "wait_adjusted_slot_score", "class_1_weight": weight, "slot_cost": slot_cost, "wait_penalty": wait_penalty, "score_value": weighted_slot_score - slot_cost * q_share - delay_cost})

    return rows


def rule_rows_for_run(row: pd.Series) -> list[dict]:
    rows = []
    for served_floor in SERVED_RATE_FLOORS:
        for gap_limit in CLASS_UTILIZATION_GAP_LIMITS:
            passes_rules = (
                row["average_utilization"] >= UTILIZATION_FLOOR
                and row["min_class_served_rate"] >= served_floor
                and row["class_utilization_gap"] <= gap_limit
            )
            rows.append(
                {
                    **row.to_dict(),
                    "utilization_floor": UTILIZATION_FLOOR,
                    "served_rate_floor": served_floor,
                    "class_utilization_gap_limit": gap_limit,
                    "passes_rules": passes_rules,
                    "utilization_if_rules_pass": row["average_utilization"] if passes_rules else np.nan,
                    "wait_score_if_rules_pass": -row["mean_offered_booking_delay"] if passes_rules else np.nan,
                }
            )
    return rows

## Matched FCFS Example

Run FCFS and strict reservation with the same seeds. This makes the comparison cleaner because both policies see the same random starting points.


In [ ]:
run_rows = []
for seed in BASE_SCENARIO["seeds"]:
    for policy, q in [("Pooled FCFS", 0), ("Strict C1 reservation", BASE_SCENARIO["reserved_slots_per_day"] )]:
        config = build_config(BASE_SCENARIO, q=q, seed=int(seed))
        result = ClinicAppointmentSimulation(config).run()
        fixed = {"policy": policy, "seed": int(seed), "Q": int(q)}
        run_rows.append(result_summary_row(result, fixed))

run_df = pd.DataFrame(run_rows)
run_df.apply(validate_accounting, axis=1)

score_df = pd.DataFrame([objective for _, row in run_df.iterrows() for objective in score_rows_for_run(row)])
rule_df = pd.DataFrame([constraint for _, row in run_df.iterrows() for constraint in rule_rows_for_run(row)])

display(run_df.groupby("policy")[["served_rate", "average_utilization", "mean_offered_booking_delay", "class_1_served_rate", "class_2_served_rate", "class_utilization_gap"]].mean().round(4))
display(score_df.head())
display(rule_df.head())

## Checks

These checks make sure the notebook is comparing sensible numbers: FCFS has no protected slots, weighted served rates stay between 0 and 1, and higher slot costs do not make a policy look better.


In [ ]:
assert (run_df.loc[run_df["policy"] == "Pooled FCFS", "Q"] == 0).all(), "FCFS must have Q = 0."
assert score_df.loc[score_df["policy"] == "Pooled FCFS", "slot_cost"].ge(0).all(), "Slot costs should be nonnegative."

weighted_served = score_df[score_df["score_name"] == "weighted_served_rate"]
assert weighted_served["score_value"].between(0, 1).all(), "Weighted served score must be in [0, 1]."

net = score_df[score_df["score_name"] == "net_priority_score"].sort_values("slot_cost")
monotone = net.groupby(["policy", "seed", "Q", "class_1_weight"])["score_value"].apply(lambda s: s.diff().dropna().le(1e-12).all())
assert monotone.all(), "Higher slot-cost penalties should not increase net priority score."

fcfs_by_cost = score_df[score_df["policy"] == "Pooled FCFS"].groupby(["score_name", "seed", "class_1_weight", "wait_penalty"])
for _, group in fcfs_by_cost:
    if group["slot_cost"].nunique() > 1:
        assert group["score_value"].nunique() == 1, "FCFS score should not change with slot cost because Q = 0."

print("Score checks passed.")

## Difference From FCFS

The table below reports strict reservation minus FCFS. Positive values mean strict reservation did better under that scoring rule. Negative values mean FCFS did better.


In [ ]:
paired = score_df.pivot_table(
    index=["seed", "score_name", "class_1_weight", "slot_cost", "wait_penalty"],
    columns="policy",
    values="score_value",
).reset_index()
paired["difference_vs_fcfs"] = paired["Strict C1 reservation"] - paired["Pooled FCFS"]

difference_summary = (
    paired.groupby(["score_name", "class_1_weight", "slot_cost", "wait_penalty"])["difference_vs_fcfs"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
difference_summary["sem"] = difference_summary["std"].fillna(0.0) / np.sqrt(difference_summary["count"])
difference_summary["ci_low"] = difference_summary["mean"] - 1.96 * difference_summary["sem"]
difference_summary["ci_high"] = difference_summary["mean"] + 1.96 * difference_summary["sem"]
difference_summary["classification"] = np.select(
    [
        (difference_summary["mean"] > 0) & (difference_summary["ci_low"] > 0),
        difference_summary["mean"] > 0,
    ],
    ["win", "possible_win"],
    default="loss",
)

display(difference_summary.sort_values(["score_name", "class_1_weight", "slot_cost", "wait_penalty"]).head(20))

In [ ]:
plot_df = difference_summary[
    (difference_summary["score_name"].isin(["weighted_served_rate", "weighted_slot_score", "net_priority_score"]))
    & (difference_summary["wait_penalty"] == 0)
    & (difference_summary["slot_cost"].isin([0.0, 0.02]))
].copy()
plot_df["label"] = plot_df.apply(lambda row: f"{row['score_name']}\nw={row['class_1_weight']}, c={row['slot_cost']}", axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df["label"], plot_df["mean"], yerr=1.96 * plot_df["sem"], capsize=3, color="#2563eb")
ax.axhline(0, color="0.25", linewidth=1)
ax.set_title("Strict reservation score difference vs FCFS")
ax.set_ylabel("mean score difference")
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()